# Time Series Forecasting - Statistical Models

### 1. Import Libraries

In [ ]:
pip install pmdarima

In [ ]:
import warnings; warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.seasonal import seasonal_decompose

import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm          # AutoARIMA — install once below if needed

### 2. Load & Clean the data

#### Import Data

```python
df = pd.read_excel(file) # if excel file
df = pd.read_csv(file)   # if csv file
```

```python
df.info()
```

#### 3.2 Datatypes


#### 3.3 Missing values (Nulls)
```python
df.dropna() # Remove rows
df.dropna(axis=1) # Remove columns
df[col] = df[col].fillna(np.mean(df[col])) # Fill with mean
```

#### 3.4 Outliers

```python
df.describe()   # descriptive statistics
filtered_df = df[(df[column] > value) # Filter outliers
```

#### 3.5 Duplicates

```python
df.drop_duplicates # Drop Duplicates
```

In [ ]:
# TODO: Load & CLean the data


### 3. Prepare the time series

**Quartely**  
Date column:  →  use `pd.PeriodIndex(year=..., quarter=..., freq='Q').to_timestamp()`  

Frequency: `'QS'` (Quarterly Start)  
Seasonal period: **4**

<br>

**Monthly**  
Date column: --> use `pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str) + '-01')`

Frequency: `'MS'`  
Seasonal period: **12**

<br>

**Daily**
Date column: `pd.to_datetime(df['date'])`
Frequency: `'D'`

Seasonal period: **7** (weekly cycle) or **365** (annual cycle)

<br>

**Hourly**
Date column: `pd.to_datetime(df['datetime'])` or `pd.to_datetime(df['date'] + ' ' + df['hour'].astype(str) + ':00')`
Frequency: `'h'`

Seasonal period: **24** (daily cycle), **168** (weekly = 24×7), or **8760** (annual = 24×365)

<br>


**Already, a 'date' variable**  --> use `df['Date'] = pd.to_datetime(df['date'])`

In [ ]:
# TODO: Build Date column (see hints above)

# TODO: Convert to time series


In [ ]:
# TODO: Info, missing values, descriptive stats


### 4. Data Pre-Processing

#### 4.1 Visualize the Data to see any meaningful patterns

```python
plt.plot(ts.index, ts.values, label='----')
plt.show()
# add title, legend, customize axes and other options as needed

decomp = seasonal_decompose(ts)
fig = decomp.plot()
```
seasonal_decompose - https://www.statsmodels.org/dev/generated/statsmodels.tsa.seasonal.seasonal_decompose.html

#### 4.2 Divide the data into training and testing sets

```python
size = 20
train = ts.iloc[:-size]
test = ts.iloc[-size:]
```


In [ ]:
# Decide on a test size



### 5. Fit All Five Models

For **each model**, explain in a comment:
- Why the parameters were chosen  
- Whether you expect this model to perform well on this data


**Exponential Smoothing**
$$
\begin{aligned}
\ell_t &= \alpha y_t + (1 - \alpha)\ell_{t-1} \\
\hat{y}_{t+1|t} &= \ell_t
\end{aligned}
$$

- $\ell_t$: level (smoothed value)  
- $\alpha \in (0,1)$: smoothing constant  
- $\hat{y}_{t+1|t}$: one-step forecast  
  
```python
ses_model = ExponentialSmoothing(train, trend=None, seasonal=None).fit(smoothing_level=0.8, optimized=True)
ses_forecast = pd.Series(ses_model.forecast(size), index=test.index, name='SES')
```



In [ ]:
# ---- 5a. SES ----


**Holt's Linear Method**
$$
\begin{aligned}
\ell_t &= \alpha y_t + (1 - \alpha)(\ell_{t-1} + b_{t-1}) \\
b_t &= \beta (\ell_t - \ell_{t-1}) + (1 - \beta)b_{t-1} \\
\hat{y}_{t+h|t} &= \ell_t + h b_t
\end{aligned}
$$

- $\ell_t$: level  
- $b_t$: trend  
- $\alpha, \beta \in (0,1)$: smoothing parameters

   
```python
holt_model = ExponentialSmoothing(train, trend='add', seasonal=None).fit(smoothing_level=0.8,smoothing_trend=0.2, optimized=True)
holt_forecast = pd.Series(ses_model.forecast(size), index=test.index, name='Holt')
```



In [ ]:
# ---- 5b. Holt ----



**Winter Holt's Method - Trend + Seasonality**
$$
\begin{aligned}
\ell_t &= \alpha (y_t - s_{t-m}) + (1 - \alpha)(\ell_{t-1} + b_{t-1}) \\
b_t &= \beta (\ell_t - \ell_{t-1}) + (1 - \beta)b_{t-1} \\
s_t &= \gamma (y_t - \ell_t) + (1 - \gamma)s_{t-m} \\
\hat{y}_{t+h|t} &= \ell_t + h b_t + s_{t+h-m\lfloor (h-1)/m \rfloor}
\end{aligned}
$$

- $m$: seasonal period (4 for quarters)  
- $\alpha, \beta, \gamma$: smoothing constants  

```python
hw_model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=4).fit(smoothing_level=0.8, smoothing_trend=0.2, smoothing_seasonal = 0.2, optimized=True)
hw_forecast = pd.Series(hw_model.forecast(size), index=test.index, name = 'Winter Holt')
```

In [ ]:
# ---- 5c. Holt-Winters ----
# (choose the right seasonal_periods!)


**ARIMA**

| Component                         | Parameter | What It Does                                           | Scope                 |
| --------------------------------- | --------- | ------------------------------------------------------ | --------------------- |
| **AR (AutoRegressive)**           | `p` | Uses past values                                       | Short-term / Seasonal |
| **I (Integrated / Differencing)** | `d` | Removes trends or seasonal drift                       | Short-term / Seasonal |
| **MA (Moving Average)**           | `q` | Uses past forecast errors                              | Short-term / Seasonal |
| **Seasonal period**               | `m` | Defines how many steps per season (e.g. 4 = quarterly) | Seasonal              |

```python
arima_model = SARIMAX(
    train,
    order=(1,1,1),
    seasonal_order=(1,1,1,4)
).fit(disp=False)

arima_forecast = pd.Series(arima_model.forecast(steps=size), index=test.index,name = 'SARIMAX')
```

In [ ]:
# ---- 5d. SARIMA ----
# (choose the right m!)


**auto-ARIMA**

```python
auto_arima = pm.auto_arima(train, seasonal=True, m=12)
print(auto_arima.summary())
auto_arima_forecast = pd.Series(auto_arima.predict(n_periods=size), index=test.index,name = 'Auto-Arima')
```

In [ ]:
# ---- 5e. auto-ARIMA ----
# (choose the right m!)



### 6. Visualize Results

```python
plt.figure(figsize=(10,4))
plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Test')
plt.plot(model_forecast.index, model_forecast, label='-----')
plt.legend()
plt.show()
```

In [ ]:
# TODO: One combined chart showing train, actual test, and all 5 forecasts



### 7. Error Metrics

```python
def evaluate(name, actual, predicted):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    return {'Model': name, 'RMSE': round(rmse, 4), 'MAE': round(mae, 4)}

# TODO: Build the results DataFrame
results = pd.DataFrame([evaluate('model',test, model_forecast), ......]).sort_values('RMSE').reset_index(drop=True)
print(results.to_string(index=False))
```

In [ ]:
# TODO: Build the results DataFrame


### 8. Interpretation Questions  *(answer in Markdown cells below each question)*

**Q1.** Which model had the lowest RMSE? Why do you think it performed best on this dataset?

**Q2.** Look at the decomposition plot. Is there evidence of trend? Seasonality? How does that relate to which models performed well?

**Q3.** How would you change the SARIMA parameters if the model performed poorly? What diagnostics would you check? - ```arima_model.plot_diagnostics()```